# R16 · Two-stage model: find each building, then grade it

Task **R16** from `Essential_task_delegation/Next_Steps_Roadmap.md`. **Decided 19 Oct: R16 runs regardless of R12.**

**Why**
- **One grade per building:** counting questions (O2) become simple counts, and DisasTeller's reports (O3) get a per-building list.
- **Clean for the benchmark:** the grader trains on DisasterM3's training images only, SAM 3 isn't trained at all, and R11 never saw a benchmark picture, so R16's output may answer benchmark questions. R12's may not.
- **A different fix for "missed as background":** buildings are found in the **before** image, where rubble or lava can't hide them (E1: 59.5% of R9's missed destroyed pixels were predicted as background).
- **Evidence:** BRIGHT (finding buildings first and then grading them beat one-step models, 67.6 against 64.9 mIoU), xView2's 1st place, and BDAChat (before and after crops per building).

**How it works**
1. **Stage 1, find buildings:** SAM 3's instance masks on the before-image as a 2×2 grid (E3's `pre_grid2`, which passed Gate G3 with building IoU 0.630 against R9's 0.591).
2. **Stage 2, grade each building:** a Siamese ResNet-18 sees a 96 px crop of the before-image and the same crop of the after-image, each with the building's mask as a 4th channel. Both crops go through the same network, and `[before, after, after − before]` features give intact, damaged or destroyed.
   - **Training buildings:** the 8-connected pieces of each class in the ground-truth masks of the 5,798 training images, with a 16 px margin. Every damaged and destroyed piece is kept, plus at most 12 intact pieces per image.
   - **Balanced grades:** each grade is drawn equally often. 10 epochs; the last epoch is the reported model.
3. **R11** (6 channels, never trained on benchmark pictures) runs on the 645 validation images with 8-view TTA. Its scores must reproduce `D4_R11.csv`, which checks the whole pipeline, and its predicted masks are saved for later tasks.

**What it reports** (E-protocol: the same 645 validation images, full resolution, dataset-level confusion matrix)

| Row | Buildings from | Grade from |
|---|---|---|
| `R11_std`, `R11_tta` | (reference) | R11's pixels |
| `R16_sam3` | SAM 3 | the grader |
| `R16_sam3+r11` | SAM 3, plus R11's building pieces that SAM 3 missed | the grader |
| `R16_vote` | as above | the average of the grader's probabilities and R11's pixel shares inside the building |

- **The grader alone on the true buildings:** object-level precision, recall and macro F1 over the validation buildings, given their true outlines. This compares with BDAChat's 0.763, but only roughly, because BDAChat used its own split.
- **Keep rule** against R11 with TTA (0.5875 / 0.2965): Destroyed IoU up by more than 0.013, with mIoU down by no more than 0.005.

**How to run on Kaggle**
1. Settings: GPU T4 ×1, internet ON, secret `HF_TOKEN` from the AbrarAlam account (R11's checkpoint is in the private repo `AbrarAlam/disasterm3-d4-R11`).
2. Inputs:
   - the DisasterM3 mirror (attached automatically);
   - **E3's output:** Add Input → Your Work → `e3-sam3-building-finding` (its notebook output). Run R16 on the account that ran E3, or upload E3's `masks/pre_grid2/` folder as a dataset. Cell 2 finds it by the folder name `pre_grid2`;
   - optional: R11's `final_model.pth` as a dataset (in a folder named `R11`, `d4_R11` or `R11-results`), if Hugging Face fails.
3. Run All once with `SMOKE_TEST = True` (16 training and 8 validation images, 1 epoch, a few minutes). Then set it to `False` and use Save & Run All, about 1 hour.
4. Outputs in `/kaggle/working/R16/`: `D4_R16.csv`, `R16_summary.csv` (also split by source), `grader_true_buildings.csv`, `instances_val.csv` (one row per stage-1 building), `grader.pth`, `history.json`, `R11_tta/` (R11's predicted masks), `config.json` and `README.md`.

In [1]:
# ── Cell 1: Environment Setup (Run-All safe) ──────────────────────────────
# segmentation-models-pytorch 0.3.4 is needed only to load R11 (its checkpoint key layout).
# The grader uses torchvision, which Kaggle already has.
import importlib.metadata as _md
import os, sys, subprocess

PINS = {
    "segmentation-models-pytorch": "0.3.4",   # baseline version (checkpoint key layout)
}

def _mismatched():
    out = []
    for pkg, want in PINS.items():
        try:
            have = _md.version(pkg)
        except _md.PackageNotFoundError:
            have = "not installed"
        if have != want:
            out.append(f"{pkg}: {have} → {want}")
    return out

if _mismatched():
    print("⏳ Installing pinned versions:")
    for _m in _mismatched():
        print(f"   {_m}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    *(f"{k}=={v}" for k, v in PINS.items()), "huggingface_hub[hf_xet]"],
                   check=True)                # pip failure stops the run here
    if _mismatched():
        raise RuntimeError(f"Pins still wrong after install: {_mismatched()}")
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch":
        # background "Save & Run All": nothing imported yet, so new versions load on first import
        print("✓ Dependencies installed (background run) — continuing without restart.")
    else:
        raise SystemExit("✓ Dependencies installed. RESTART THE KERNEL NOW "
                         "(Run → Restart & clear cell outputs), then Run All again.")
else:
    print("✓ All pinned versions already installed — proceeding.")

import torch
print(f"  torch {torch.__version__}, python {sys.version.split()[0]}")


⏳ Installing pinned versions:
   segmentation-models-pytorch: not installed → 0.3.4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.5/109.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 65.7 MB/s eta 0:00:00
✓ Dependencies installed (background run) — continuing without restart.
  torch 2.11.0+cu128, python 3.13.15


In [2]:
# ── Cell 2: Configuration and inputs ──────────────────────────────────────
import glob
import json
import os
import random
import re
import time
from pathlib import Path

import numpy as np
import torch
from huggingface_hub import hf_hub_download

DATA_ROOT = Path("/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct")
MANIFEST_PATH = DATA_ROOT / "train_release.json"
OUT_DIR = Path("/kaggle/working/R16")
CACHE_DIR = Path("/kaggle/temp/r16_crops")   # training crops (about 8 GB), not saved as output

NUM_CLASSES = 4                     # 0=Background, 1=Intact, 2=Damaged, 3=Destroyed
CLASS_NAMES = ["Background", "Intact", "Damaged", "Destroyed"]
GRADES = CLASS_NAMES[1:]            # the grader's classes 0-2 are mask classes 1-3
SMOKE_TEST = False                  # True = 16 training and 8 validation images, 1 epoch

CROP = 96                           # crop side after resizing (the roadmap said 128; 96 keeps the cache near 8 GB)
MARGIN = 16                         # context around each building's box, in pixels
MIN_AREA = 20                       # building pieces smaller than this are skipped (E1's VOTE_MIN_AREA)
MAX_INTACT_PER_IMAGE = 12           # damaged and destroyed training pieces are all kept
EPOCHS, BATCH, LR, WEIGHT_DECAY = 10, 256, 3e-4, 1e-4
SEED = 42
NUM_WORKERS = 4

# The U-Net for the footprint merge, the vote and the reference rows. R11 never trained on benchmark
# pictures, so R16's output may answer benchmark questions. For the O1 score only, set "R12" here once
# it has run, and copy its two rows from D4_R12.csv into EXPECTED_UNET.
UNET = "R11"
EXPECTED_UNET = {"std": (0.5811, 0.2911), "tta": (0.5875, 0.2965)}   # D4_R11.csv: (mIoU, Destroyed IoU)
HF_USER = "AbrarAlam"               # checkpoints: {HF_USER}/disasterm3-d4-<run>/final_model.pth
KEEP_RULE = {"min_destroyed_gain": 0.013, "max_miou_drop": 0.005}     # against the U-Net with TTA

if SMOKE_TEST:
    EPOCHS, BATCH = 1, 32

def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

def find_checkpoint(name):
    """As in E1: a /kaggle/input folder named R11 / d4_R11 / R11-results, then Hugging Face."""
    wanted = {name.lower(), f"d4_{name}".lower(), f"{name}-results".lower()}
    hits = [p for p in glob.glob("/kaggle/input/**/*.pth", recursive=True)
            if any(part.lower() in wanted for part in Path(p).parts)]
    hits.sort(key=lambda p: Path(p).name != "final_model.pth")   # prefer final_model.pth
    if hits:
        return hits[0]
    repo = f"{HF_USER}/disasterm3-d4-{name}"
    errors = []
    for filename in ("final_model.pth", "last.pth"):   # last.pth holds the same epoch-40 weights
        try:
            return hf_hub_download(repo, filename, token=hf_token())
        except Exception as e:
            errors.append(f"{filename}: {type(e).__name__}")
    raise FileNotFoundError(f"No checkpoint for {name}: nothing under /kaggle/input and {repo} gave {errors}. "
                            f"Add a Kaggle dataset with {name}/final_model.pth.")

# Stage 1: E3's SAM 3 instance maps of the before-image, 2x2 grid (16-bit PNG, 0 = background)
e3_dirs = sorted(glob.glob("/kaggle/input/**/pre_grid2", recursive=True))
if not e3_dirs:
    raise FileNotFoundError("E3's output isn't attached: add the e3-sam3-building-finding notebook output, "
                            "or a dataset holding its masks/pre_grid2/ folder (see Cell 0).")
SAM3_DIR = Path(e3_dirs[0])
UNET_CKPT = find_checkpoint(UNET)

OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"✓ R16 two-stage model{'  [SMOKE TEST]' if SMOKE_TEST else ''}")
print(f"  Stage 1 (SAM 3): {SAM3_DIR} ({len(list(SAM3_DIR.glob('*.png')))} maps)")
print(f"  U-Net {UNET}: {UNET_CKPT}")
print(f"  Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}; outputs: {OUT_DIR}")


final_model.pth: reconstructing file:   0%|          |  0.00B /  130MB            

final_model.pth: downloading bytes:           |  0.00B            

✓ R16 two-stage model
  Stage 1 (SAM 3): /kaggle/input/notebooks/abrarmohammedtanzim/e3-sam3-building-finding/E3/masks/pre_grid2 (645 maps)
  U-Net R11: /root/.cache/huggingface/hub/models--AbrarAlam--disasterm3-d4-R11/snapshots/31974dacfee2125a66639858433a113704edf9b3/final_model.pth
  Device: Tesla T4; outputs: /kaggle/working/R16


## 1. Data: the same 6,443 pairs and the same 645 validation images as every D4 run
Cell 3 is copied verbatim from R11, so the split matches R1–R12 exactly and every pair keeps its before-image.

In [3]:
# ── Cell 3: Load manifest and build the 4-class masks (verbatim from R11's Cell 4) ──
import cv2
import numpy as np
import os
from collections import Counter, defaultdict
from pathlib import Path
import json

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"Loaded {len(raw_data):,} entries from manifest")

# ── Filter: ONLY Building Damage Assessment segmentation entries ──
bda_entries = [
    e for e in raw_data
    if e.get("task") == "Referring Expression Segmentation"
    and e.get("cls_description") == "Building Damage Assessment"
]
print(f"Building Damage Assessment entries: {len(bda_entries):,}")

# ── Path resolution (same logic as before) ──
def resolve_path(rel_path):
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    candidates = [
        DATA_ROOT / rel_path,
        DATA_ROOT / "train_images" / rel_path,
        DATA_ROOT / "train_images" / "train_images" / filename,
        DATA_ROOT / "masks" / rel_path,
        DATA_ROOT / "masks" / "masks" / filename,
    ]
    for c in candidates:
        if c.exists():
            return str(c)
    return None

# ── Group entries by underlying post-disaster image ──
by_image = defaultdict(dict)  # post_image_path -> {folder_type: mask_path}

for e in bda_entries:
    post_img = e.get("post_image_path", "")
    if e.get("image_type") != "Optical":   # skip SAR duplicates, keep optical only
        continue
    mask_rel = e.get("ground_truth", "")
    folder = Path(mask_rel.replace("\\", "/")).parent.name  # e.g. train_building_intact_mask
    resolved = resolve_path(mask_rel)
    if resolved:
        by_image[post_img][folder] = resolved
        by_image[post_img]["post_image_path"] = post_img
        by_image[post_img]["pre_image_path"] = e.get("pre_image_path")   # R11: the before-image

print(f"Unique images with at least one mask: {len(by_image):,}")

# ── Build combined (image, mask) pairs ──
FOLDER_TO_CLASS = {
    "train_building_intact_mask": 1,
    "train_building_damaged_mask": 2,
    "train_building_destroyed_mask": 3,   # use 2 here instead if you want to collapse into "damaged"
}

# ── FIX: Create temporary directory on Kaggle hard drive for masks ──
os.makedirs("/tmp/combined_masks", exist_ok=True)

pairs = []
skipped_img = 0

for post_img, mask_dict in by_image.items():
    img_path = resolve_path(post_img)
    if not img_path:
        skipped_img += 1
        continue

    combined_mask = None
    for folder, class_id in FOLDER_TO_CLASS.items():
        mask_path = mask_dict.get(folder)
        if mask_path is None:
            continue
        m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if m is None:
            continue
        if combined_mask is None:
            combined_mask = np.zeros_like(m, dtype=np.uint8)
        combined_mask[m > 0] = class_id   # later classes overwrite earlier if they overlap

    if combined_mask is None:
        continue  # no valid masks found for this image at all

    # ── FIX: Save mask to hard drive instead of RAM ──
    mask_filename = Path(img_path).name
    save_path = f"/tmp/combined_masks/{mask_filename}"
    cv2.imwrite(save_path, combined_mask)

    pairs.append({
        "image_path": img_path,
        "mask_path": save_path,   # Store the string path, NOT the array!
        "pre_image_path": resolve_path(mask_dict.get("pre_image_path")),   # R11: None if not found
    })

print(f"\n✓ Built {len(pairs):,} valid (image, mask) pairs")
print(f"  Skipped (image not found): {skipped_img:,}")
print(f"  Before-image not found: {sum(p['pre_image_path'] is None for p in pairs):,} (must be 0 for R11)")

# Sanity check
if pairs:
    print(f"\n  Sample: {pairs[0]['image_path']}")
    print(f"  Before: {pairs[0]['pre_image_path']}")
    print(f"  Mask saved at: {pairs[0]['mask_path']}")


Loaded 92,968 entries from manifest
Building Damage Assessment entries: 14,531
Unique images with at least one mask: 6,443

✓ Built 6,443 valid (image, mask) pairs
  Skipped (image not found): 0
  Before-image not found: 0 (must be 0 for R11)

  Sample: /kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct/train_images/train_images/bata_explosion_post_0.png
  Before: /kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct/train_images/train_images/bata_explosion_pre_0.png
  Mask saved at: /tmp/combined_masks/bata_explosion_post_0.png


In [4]:
# ── Cell 4: Validation split (identical to every D4 run) and event names ──
from sklearn.model_selection import train_test_split

train_pairs, val_pairs = train_test_split(pairs, test_size=0.1, random_state=42)
if SMOKE_TEST:   # 16 training images, those with destroyed pixels first, so every grade gets crops
    with_destroyed = [p for p in train_pairs[:400]
                      if (cv2.imread(p["mask_path"], cv2.IMREAD_GRAYSCALE) == 3).any()]
    train_pairs, val_pairs = (with_destroyed + train_pairs)[:16], val_pairs[:8]
missing_pre = sum(p["pre_image_path"] is None for p in train_pairs + val_pairs)
assert missing_pre == 0, f"{missing_pre} pairs have no before-image"

XBD_NAME = re.compile(r"^(.+?)_(\d{8})_post_disaster")
OTHER_NAME = re.compile(r"^(.+?)_post")

def event_and_source(image_path):
    """Same rule as E1: 'hurricane_florence_00000261_post_disaster.png' -> ('hurricane_florence', 'xBD')."""
    name = Path(image_path).name
    m = XBD_NAME.match(name)
    if m:
        return m.group(1), "xBD"
    m = OTHER_NAME.match(name)
    return re.sub(r"\d+$", "", m.group(1) if m else Path(name).stem), "other"

events = [event_and_source(p["image_path"]) for p in val_pairs]
n_xbd = sum(1 for _, source in events if source == "xBD")
print(f"✓ Training images: {len(train_pairs):,}. Validation images: {len(val_pairs)} ({n_xbd} xBD, "
      f"{len(val_pairs) - n_xbd} other). Expected 5,798 and 645 (560 xBD, 85 other) when SMOKE_TEST is False.")


✓ Training images: 5,798. Validation images: 645 (560 xBD, 85 other). Expected 5,798 and 645 (560 xBD, 85 other) when SMOKE_TEST is False.


## 2. One crop per building
Every building, whether it comes from the ground truth (training, and the grader-alone check) or from stage 1 (validation), becomes the same thing: a square crop around its box with a 16 px margin, resized to 96 px, holding the before-image (3 channels), the after-image (3) and the building's own mask (1). Ground-truth buildings are the 8-connected pieces of one class, so each training piece has exactly one grade.

In [5]:
# ── Cell 5: Crop helpers ────────────────────────────────────────────────────
from scipy import ndimage

def read_triplet(pair):
    """After-image and before-image (RGB, same size) and the 4-class mask, as in the D4 dataset class."""
    post, pre = cv2.imread(pair["image_path"]), cv2.imread(pair["pre_image_path"])
    gt = cv2.imread(pair["mask_path"], cv2.IMREAD_GRAYSCALE)
    if post is None or pre is None or gt is None:
        return None
    post, pre = cv2.cvtColor(post, cv2.COLOR_BGR2RGB), cv2.cvtColor(pre, cv2.COLOR_BGR2RGB)
    h, w = post.shape[:2]
    if pre.shape != post.shape:
        pre = cv2.resize(pre, (w, h), interpolation=cv2.INTER_LINEAR)
    if gt.shape != (h, w):
        gt = cv2.resize(gt, (w, h), interpolation=cv2.INTER_NEAREST)
    return post, pre, gt

def square_box(sl, h, w):
    """Square window around a building's (row, column) slices, MARGIN px wider on each side, kept inside the image."""
    y0, y1, x0, x1 = sl[0].start, sl[0].stop, sl[1].start, sl[1].stop
    side = min(max(y1 - y0, x1 - x0) + 2 * MARGIN, h, w)
    cy, cx = (y0 + y1) / 2, (x0 + x1) / 2
    top = int(round(min(max(cy - side / 2, 0), h - side)))
    left = int(round(min(max(cx - side / 2, 0), w - side)))
    return top, left, side

def building_crops(pre, post, labels, ids):
    """labels: int map of building ids (0 = none). Returns uint8 (len(ids), 7, CROP, CROP):
    before RGB, after RGB, and 255 where the pixel belongs to that building."""
    h, w = labels.shape
    slices = ndimage.find_objects(labels)
    out = np.empty((len(ids), 7, CROP, CROP), np.uint8)
    for n, lid in enumerate(ids):
        top, left, side = square_box(slices[lid - 1], h, w)
        win = (slice(top, top + side), slice(left, left + side))
        out[n, 0:3] = cv2.resize(pre[win], (CROP, CROP), interpolation=cv2.INTER_LINEAR).transpose(2, 0, 1)
        out[n, 3:6] = cv2.resize(post[win], (CROP, CROP), interpolation=cv2.INTER_LINEAR).transpose(2, 0, 1)
        out[n, 6] = cv2.resize((labels[win] == lid).astype(np.uint8) * 255, (CROP, CROP),
                               interpolation=cv2.INTER_NEAREST)
    return out

def gt_pieces(gt):
    """Ground-truth buildings: 8-connected pieces of each class 1-3 with at least MIN_AREA pixels.
    Yields (class, label map, kept ids)."""
    for cls in (1, 2, 3):
        n, labels, stats, _ = cv2.connectedComponentsWithStats((gt == cls).astype(np.uint8), connectivity=8)
        ids = [i for i in range(1, n) if stats[i, cv2.CC_STAT_AREA] >= MIN_AREA]
        if ids:
            yield cls, labels, ids

# Self-check: a box at the image corner stays inside, and each crop marks only its own building
_lab = np.zeros((200, 300), np.int32)
_lab[0:10, 0:20] = 1
_lab[100:150, 200:260] = 2
assert square_box(ndimage.find_objects(_lab)[0], 200, 300) == (0, 0, 52)
_crops = building_crops(np.zeros((200, 300, 3), np.uint8), np.zeros((200, 300, 3), np.uint8), _lab, [1, 2])
assert _crops.shape == (2, 7, CROP, CROP) and _crops[1, 6].max() == 255 and _crops[0, 6].mean() < 255
_gt = np.zeros((200, 300), np.uint8)
_gt[0:10, 0:20], _gt[100:150, 200:260], _gt[190:192, 0:2] = 3, 1, 2      # the 4-pixel piece is below MIN_AREA
assert [(c, len(i)) for c, _, i in gt_pieces(_gt)] == [(1, 1), (3, 1)]
print("✓ Crop helpers ready (self-checks passed)")


✓ Crop helpers ready (self-checks passed)


In [6]:
# ── Cell 6: Crops of the training buildings → one uint8 file on disk ────────
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

class ImageCrops(Dataset):
    """All ground-truth building crops of one image, with their grades (0-2)."""

    def __init__(self, pairs_subset, intact_cap=None):
        self.pairs, self.intact_cap = pairs_subset, intact_cap

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        crops, grades = [np.zeros((0, 7, CROP, CROP), np.uint8)], []
        data = read_triplet(self.pairs[idx])
        if data is not None:
            post, pre, gt = data
            rng = np.random.default_rng(SEED + idx)    # the same intact sample whichever worker reads the image
            for cls, labels, ids in gt_pieces(gt):
                if cls == 1 and self.intact_cap and len(ids) > self.intact_cap:
                    ids = sorted(rng.choice(ids, self.intact_cap, replace=False).tolist())
                crops.append(building_crops(pre, post, labels, ids))
                grades += [cls - 1] * len(ids)
        return np.concatenate(crops), np.array(grades, np.int64)

def build_cache(pairs_subset, name, intact_cap=None):
    """Writes every crop to CACHE_DIR/<name>.u8 and returns that file and the grades."""
    path, grades = CACHE_DIR / f"{name}.u8", []
    loader = DataLoader(ImageCrops(pairs_subset, intact_cap), batch_size=None, num_workers=NUM_WORKERS)
    with open(path, "wb") as f:
        for crops, g in tqdm(loader, desc=f"Crops ({name})"):
            f.write(np.asarray(crops, np.uint8).tobytes())
            grades.append(np.asarray(g, np.int64))
    return path, np.concatenate(grades)

t0 = time.time()
TRAIN_CACHE, train_grades = build_cache(train_pairs, "train", MAX_INTACT_PER_IMAGE)
counts = np.bincount(train_grades, minlength=3)
print(f"✓ {len(train_grades):,} training buildings in {(time.time() - t0) / 60:.1f} min "
      f"({TRAIN_CACHE.stat().st_size / 1e9:.1f} GB): " + ", ".join(f"{g} {c:,}" for g, c in zip(GRADES, counts)))
assert TRAIN_CACHE.stat().st_size == len(train_grades) * 7 * CROP * CROP, "cache size and grade count disagree"
assert counts.min() > 0, "a grade has no training buildings"


Crops (train):   0%|          | 0/5798 [00:00<?, ?it/s]

✓ 102,716 training buildings in 4.6 min (6.6 GB): Intact 42,814, Damaged 41,781, Destroyed 18,121


## 3. Stage 2: the Siamese grader
One ResNet-18 (ImageNet weights) reads the before crop and the after crop separately; its first layer takes a 4th channel, the building's mask. The head sees `[before, after, after − before]`. Training draws each grade equally often, flips and rotates the crops, and keeps the last epoch (no checkpoint is picked on validation).

In [7]:
# ── Cell 7: Stage 2, the Siamese grader ────────────────────────────────────
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import WeightedRandomSampler

class CropSet(Dataset):
    """Crops from a cache file. The memmap opens inside each worker, so it is never pickled."""

    def __init__(self, path, grades, augment):
        self.path, self.grades, self.augment, self.data = path, grades, augment, None

    def __len__(self):
        return len(self.grades)

    def __getitem__(self, i):
        if self.data is None:
            self.data = np.memmap(self.path, np.uint8, "r", shape=(len(self.grades), 7, CROP, CROP))
        x = np.array(self.data[i])
        if self.augment:   # the same flip and rotation for the before crop, the after crop and the mask
            x = np.rot90(x, random.randrange(4), axes=(1, 2))
            if random.random() < 0.5:
                x = x[:, :, ::-1]
        return torch.from_numpy(np.ascontiguousarray(x)), int(self.grades[i])

class SiameseGrader(nn.Module):
    """A shared ResNet-18 reads the before and the after crop (RGB + building mask each);
    [f_before, f_after, f_after - f_before] -> intact, damaged, destroyed."""

    def __init__(self):
        super().__init__()
        net = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
        rgb = net.conv1.weight.data
        net.conv1 = nn.Conv2d(4, 64, kernel_size=7, stride=2, padding=3, bias=False)
        net.conv1.weight.data = torch.cat([rgb, rgb.mean(1, keepdim=True)], 1)   # the mask channel starts as grey
        net.fc = nn.Identity()
        self.encoder = net
        self.head = nn.Sequential(nn.Linear(3 * 512, 256), nn.ReLU(inplace=True), nn.Dropout(0.2), nn.Linear(256, 3))

    def forward(self, before, after):
        fb, fa = self.encoder(before), self.encoder(after)
        return self.head(torch.cat([fb, fa, fa - fb], 1))

MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1) * 255
STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1) * 255

def to_inputs(x):
    """uint8 (B, 7, S, S) -> normalised before and after tensors (B, 4, S, S); the mask becomes -1 / +1."""
    x = x.to(DEVICE, non_blocking=True).float()
    mask = x[:, 6:7] / 127.5 - 1
    return torch.cat([(x[:, 0:3] - MEAN) / STD, mask], 1), torch.cat([(x[:, 3:6] - MEAN) / STD, mask], 1)

@torch.no_grad()
def grade(crops):
    """uint8 crops (N, 7, S, S) -> grade probabilities (N, 3)."""
    grader.eval()
    out = []
    for i in range(0, len(crops), 512):
        before, after = to_inputs(torch.from_numpy(np.ascontiguousarray(crops[i:i + 512])))
        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            out.append(torch.softmax(grader(before, after).float(), 1).cpu())
    return torch.cat(out).numpy() if out else np.zeros((0, 3), np.float32)

# ponytail: balanced grades favour damaged/destroyed recall over precision; if intact buildings get
# over-graded, add a logit adjustment by the training class shares (Menon et al., ICLR 2021)
train_set = CropSet(TRAIN_CACHE, train_grades, augment=True)
weights = torch.tensor(1.0 / counts[train_grades], dtype=torch.double)
sampler = WeightedRandomSampler(weights, num_samples=len(train_set), replacement=True,
                                generator=torch.Generator().manual_seed(SEED))
loader = DataLoader(train_set, batch_size=BATCH, sampler=sampler, num_workers=NUM_WORKERS,
                    pin_memory=True, drop_last=True, persistent_workers=True)

grader = SiameseGrader().to(DEVICE)
optimizer = torch.optim.AdamW(grader.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * len(loader))
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")

history, t0 = [], time.time()
for epoch in range(EPOCHS):
    grader.train()
    total, correct, seen = 0.0, 0, 0
    for x, y in tqdm(loader, desc=f"Grader {epoch + 1}/{EPOCHS}", leave=False):
        before, after = to_inputs(x)
        y = y.to(DEVICE)
        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            logits = grader(before, after)
        loss = F.cross_entropy(logits.float(), y)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total += loss.item() * len(y)
        correct += int((logits.argmax(1) == y).sum())
        seen += len(y)
    history.append({"epoch": epoch + 1, "loss": total / max(seen, 1), "balanced_train_accuracy": correct / max(seen, 1)})
    print(f"Ep {epoch + 1:>2}/{EPOCHS} | loss {history[-1]['loss']:.4f} | train accuracy (balanced draws) "
          f"{history[-1]['balanced_train_accuracy']:.3f} | {(time.time() - t0) / 60:.1f} min")
GRADER_MIN = (time.time() - t0) / 60
torch.save(grader.state_dict(), OUT_DIR / "grader.pth")
print(f"✓ Grader trained in {GRADER_MIN:.1f} min; the last epoch is saved as grader.pth")


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 205MB/s]


Grader 1/10:   0%|          | 0/401 [00:00<?, ?it/s]

Ep  1/10 | loss 0.5128 | train accuracy (balanced draws) 0.790 | 0.9 min


Grader 2/10:   0%|          | 0/401 [00:00<?, ?it/s]

...

Ep  8/10 | loss 0.2462 | train accuracy (balanced draws) 0.903 | 7.6 min


Grader 9/10:   0%|          | 0/401 [00:00<?, ?it/s]

Ep  9/10 | loss 0.2274 | train accuracy (balanced draws) 0.911 | 8.6 min


Grader 10/10:   0%|          | 0/401 [00:00<?, ?it/s]

Ep 10/10 | loss 0.2202 | train accuracy (balanced draws) 0.914 | 9.6 min
✓ Grader trained in 9.6 min; the last epoch is saved as grader.pth


## 4. Validation: the U-Net, the grader on the true buildings, and the two-stage maps
For each of the 645 validation images:
1. **R11** predicts with and without TTA (the D4 code), and its TTA masks are saved.
2. **The grader on the true buildings:** every ground-truth piece is graded, for the object-level scores.
3. **Stage 1:** SAM 3's instances, plus R11's building pieces that SAM 3 missed (at least 20 px). The grader grades each one, and the three maps in Cell 0's table are painted and scored.

In [8]:
# ── Cell 8: The U-Net (the D4 prediction code) and the E-protocol metrics ───
import segmentation_models_pytorch as smp

def load_unet(path):
    """D4 checkpoint: U-Net + ResNet-50, 6 input channels (after-image, then before-image), 4 classes."""
    state = torch.load(path, map_location=DEVICE, weights_only=False)
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]
    model = smp.Unet(encoder_name="resnet50", encoder_weights=None, in_channels=6, classes=NUM_CLASSES)
    model.load_state_dict(state)
    return model.to(DEVICE).eval()

IMNET_MEAN = np.array([0.485, 0.456, 0.406], np.float32) * 255
IMNET_STD = np.array([0.229, 0.224, 0.225], np.float32) * 255

def unet_input(post, pre):
    """As R11's val_transform: ImageNet normalisation, channels 0-2 after-image, 3-5 before-image."""
    x = np.concatenate([(post.astype(np.float32) - IMNET_MEAN) / IMNET_STD,
                        (pre.astype(np.float32) - IMNET_MEAN) / IMNET_STD], 2)
    return torch.from_numpy(x.transpose(2, 0, 1).copy()).unsqueeze(0).to(DEVICE)

def _forward_once(model, x):
    """Logits at x's own HxW: pad to a multiple of 32, exactly as in the D4 notebooks."""
    h, w = x.shape[-2:]
    ph, pw = (32 - h % 32) % 32, (32 - w % 32) % 32
    if ph or pw:
        x = F.pad(x, (0, pw, 0, ph), mode="reflect")
    return model(x)[..., :h, :w].float()

# The D4 notebooks' 8 dihedral views: rotation 0/90/180/270 × (no flip, horizontal flip)
TTA_VIEWS = [(k, flip) for flip in (False, True) for k in range(4)]

def _tta_fwd(x, k, flip):
    x = torch.rot90(x, k, dims=(2, 3))
    return torch.flip(x, dims=(3,)) if flip else x

def _tta_inv(y, k, flip):
    if flip:
        y = torch.flip(y, dims=(3,))
    return torch.rot90(y, -k, dims=(2, 3))

@torch.no_grad()
def predict_std_and_tta(model, x):
    """One pass over the 8 views (as in E1). The identity view (k=0, no flip) is the standard prediction."""
    prob_sum, pred_std = 0, None
    with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
        for k, flip in TTA_VIEWS:
            logits = _forward_once(model, _tta_fwd(x, k, flip))
            if k == 0 and not flip:
                pred_std = logits.argmax(1)
            prob_sum = prob_sum + torch.softmax(_tta_inv(logits, k, flip), dim=1)
    pred_tta = (prob_sum / len(TTA_VIEWS)).argmax(1)
    return (pred_std[0].cpu().numpy().astype(np.uint8),
            pred_tta[0].cpu().numpy().astype(np.uint8))

def confusion(gt, pred):
    """4×4 pixel counts, rows = truth, columns = prediction."""
    idx = gt.astype(np.int64).ravel() * NUM_CLASSES + pred.astype(np.int64).ravel()
    return np.bincount(idx, minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)

def metrics(cm):
    """Dataset-level metrics from a confusion matrix (E1's formulas), plus building IoU (classes 1-3 merged)."""
    cm = cm.astype(np.float64)
    eps = 1e-8
    tp = np.diag(cm)
    fp = cm.sum(0) - tp
    fn = cm.sum(1) - tp
    iou = tp / (tp + fp + fn + eps)
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    building_tp = cm[1:, 1:].sum()
    building_iou = building_tp / (building_tp + cm[0, 1:].sum() + cm[1:, 0].sum() + eps)
    return {"miou": float(iou.mean()), "iou": iou, "precision": precision, "recall": recall,
            "f1": 2 * precision * recall / (precision + recall + eps),
            "f2": 5 * precision * recall / (4 * precision + recall + eps),
            "building_iou": float(building_iou)}

_toy = np.array([[0, 1], [2, 3]], np.uint8)
assert (metrics(confusion(_toy, _toy))["iou"] > 0.99).all()
unet = load_unet(UNET_CKPT)
print(f"✓ {UNET} loaded from {UNET_CKPT} (metric self-check passed)")


✓ R11 loaded from /root/.cache/huggingface/hub/models--AbrarAlam--disasterm3-d4-R11/snapshots/31974dacfee2125a66639858433a113704edf9b3/final_model.pth (metric self-check passed)


In [9]:
# ── Cell 9: Validation, image by image ─────────────────────────────────────
VARIANTS = [f"{UNET}_std", f"{UNET}_tta", "R16_sam3", f"R16_sam3+{UNET.lower()}", "R16_vote"]
SOURCES = ("all", "xBD", "other")
cms = {v: {s: np.zeros((NUM_CLASSES, NUM_CLASSES), np.int64) for s in SOURCES} for v in VARIANTS}
true_cm = np.zeros((3, 3), np.int64)        # the grader on the true buildings: rows truth, columns grade
inst_rows, missing_sam = [], []
pred_dir = OUT_DIR / f"{UNET}_tta"
pred_dir.mkdir(exist_ok=True)

t0 = time.time()
for pair, (event, source) in tqdm(list(zip(val_pairs, events)), desc="Validation"):
    data = read_triplet(pair)
    if data is None:
        continue
    post, pre, gt = data
    h, w = gt.shape
    stem = Path(pair["image_path"]).stem

    # 1) The U-Net, standard and 8-view TTA; its TTA masks are kept for later tasks
    pred_std, pred_tta = predict_std_and_tta(unet, unet_input(post, pre))
    cv2.imwrite(str(pred_dir / f"{stem}.png"), pred_tta)

    # 2) The grader on the true buildings (object level)
    for cls, labels, ids in gt_pieces(gt):
        np.add.at(true_cm[cls - 1], grade(building_crops(pre, post, labels, ids)).argmax(1), 1)

    # 3) Stage 1: SAM 3's instances, then the U-Net's building pieces that SAM 3 missed
    sam = cv2.imread(str(SAM3_DIR / f"{stem}.png"), cv2.IMREAD_UNCHANGED)
    if sam is None:
        missing_sam.append(stem)
        sam = np.zeros((h, w), np.uint16)
    if sam.shape != (h, w):
        sam = cv2.resize(sam, (w, h), interpolation=cv2.INTER_NEAREST)
    ids = sam.astype(np.int32)
    n_sam = int(ids.max())
    n, extra, stats, _ = cv2.connectedComponentsWithStats(((pred_tta > 0) & (ids == 0)).astype(np.uint8),
                                                          connectivity=8)
    keep = stats[:, cv2.CC_STAT_AREA] >= MIN_AREA
    keep[0] = False
    new_id = np.zeros(n, np.int32)
    new_id[keep] = n_sam + 1 + np.arange(int(keep.sum()))
    ids = np.where(extra > 0, new_id[extra], ids)          # ids above n_sam are the U-Net's pieces

    # 4) Grade every stage-1 building, then paint the three maps
    present = np.unique(ids)
    present = present[present > 0]
    size = int(ids.max()) + 1
    probs = np.zeros((size, 3), np.float32)
    if len(present):
        probs[present] = grade(building_crops(pre, post, ids, present))
    flat = ids.ravel().astype(np.int64) * NUM_CLASSES
    unet_px = np.bincount(flat + pred_tta.ravel(), minlength=size * NUM_CLASSES).reshape(size, NUM_CLASSES)
    gt_px = np.bincount(flat + gt.ravel(), minlength=size * NUM_CLASSES).reshape(size, NUM_CLASSES)
    unet_bld = unet_px[:, 1:].astype(np.float32)
    unet_sum = unet_bld.sum(1, keepdims=True)
    shares = np.where(unet_sum > 0, unet_bld / np.maximum(unet_sum, 1), probs)   # no U-Net building pixel: grader only
    by_grader = probs.argmax(1).astype(np.uint8) + 1
    by_vote = (0.5 * (probs + shares)).argmax(1).astype(np.uint8) + 1
    by_grader[0] = by_vote[0] = 0                  # id 0 is background
    sam_only = by_grader.copy()
    sam_only[n_sam + 1:] = 0

    preds = {f"{UNET}_std": pred_std, f"{UNET}_tta": pred_tta, "R16_sam3": sam_only[ids],
             f"R16_sam3+{UNET.lower()}": by_grader[ids], "R16_vote": by_vote[ids]}
    for v, p in preds.items():
        cm = confusion(gt, p)
        cms[v]["all"] += cm
        cms[v][source] += cm

    for lid in present:
        inst_rows.append({"image": f"{stem}.png", "event": event, "source": source, "id": int(lid),
                          "stage1": "sam3" if lid <= n_sam else UNET.lower(), "area_px": int(gt_px[lid].sum()),
                          **{f"p_{g.lower()}": round(float(probs[lid, i]), 4) for i, g in enumerate(GRADES)},
                          **{f"{UNET.lower()}_{c.lower()}_px": int(unet_px[lid, i]) for i, c in enumerate(CLASS_NAMES)},
                          **{f"gt_{c.lower()}_px": int(gt_px[lid, i]) for i, c in enumerate(CLASS_NAMES)}})

if missing_sam:
    print(f"⚠ {len(missing_sam)} validation images had no SAM 3 map (they got the U-Net's pieces only), "
          f"e.g. {missing_sam[:3]}")
assert len(missing_sam) <= 0.05 * len(val_pairs), "Most SAM 3 maps are missing: is SAM3_DIR E3's pre_grid2 folder?"
VAL_MIN = (time.time() - t0) / 60
print(f"✓ {len(val_pairs)} validation images in {VAL_MIN:.1f} min; {len(inst_rows):,} stage-1 buildings graded")


Validation:   0%|          | 0/645 [00:00<?, ?it/s]

✓ 645 validation images in 8.7 min; 57,991 stage-1 buildings graded


## 5. Results, the sanity check, the keep rule and the output files

In [10]:
# ── Cell 10: Results, the sanity check, the keep rule and the output files ──
import csv
import pandas as pd

def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:              # `tabulate` missing: fall back to a plain-text table
        return "```\n" + df.to_string(index=False) + "\n```"

# 1) Pixel scores per variant, overall and per source
rows = []
for v in VARIANTS:
    for s in SOURCES:
        m = metrics(cms[v][s])
        rows.append({"variant": v, "source": s, "miou": round(m["miou"], 4),
                     **{f"{c.lower()}_iou": round(float(m["iou"][i]), 4) for i, c in enumerate(CLASS_NAMES)},
                     "building_iou": round(m["building_iou"], 4),
                     "destroyed_precision": round(float(m["precision"][3]), 4),
                     "destroyed_recall": round(float(m["recall"][3]), 4)})
summary = pd.DataFrame(rows)
summary.to_csv(OUT_DIR / "R16_summary.csv", index=False)
overall = summary[summary["source"] == "all"].drop(columns="source")
print(overall.to_string(index=False))

# 2) Sanity check: the U-Net must reproduce its D4 CSV
ok = all(abs(metrics(cms[f"{UNET}_{mode}"]["all"])["miou"] - exp[0]) < 5e-4
         and abs(metrics(cms[f"{UNET}_{mode}"]["all"])["iou"][3] - exp[1]) < 5e-4
         for mode, exp in EXPECTED_UNET.items())
sanity = "n/a (smoke test)" if SMOKE_TEST else ("yes" if ok else "NO")
print(f"\n{UNET} reproduces its D4 CSV: {sanity}")
if sanity == "NO":
    print("⚠ Check the checkpoint (wrong run or not the last epoch) before trusting the comparison.")

# 3) The grader on the true validation buildings (object level, rows = truth)
tp = np.diag(true_cm).astype(np.float64)
precision = tp / np.maximum(true_cm.sum(0), 1)
recall = tp / np.maximum(true_cm.sum(1), 1)
f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-8)
true_df = pd.DataFrame({"grade": GRADES, "buildings": true_cm.sum(1), "precision": precision.round(4),
                        "recall": recall.round(4), "f1": f1.round(4),
                        **{f"graded_{g.lower()}": true_cm[:, i] for i, g in enumerate(GRADES)}})
true_df.to_csv(OUT_DIR / "grader_true_buildings.csv", index=False)
macro_f1 = round(float(f1.mean()), 4)
print(f"\nThe grader on the true validation buildings:\n{true_df.to_string(index=False)}")
print(f"Macro F1 {macro_f1:.4f} (BDAChat reports 0.763 on its own split, so the comparison is rough)")

# 4) The keep rule against the U-Net with TTA, and the Phase Gate
base = metrics(cms[f"{UNET}_tta"]["all"])
verdicts = {}
print(f"\nKeep rule against {UNET} TTA (Destroyed IoU up by more than {KEEP_RULE['min_destroyed_gain']}, "
      f"mIoU down by at most {KEEP_RULE['max_miou_drop']}):")
for v in VARIANTS[2:]:
    m = metrics(cms[v]["all"])
    d_miou, d_destroyed = m["miou"] - base["miou"], float(m["iou"][3] - base["iou"][3])
    verdicts[v] = bool(d_destroyed > KEEP_RULE["min_destroyed_gain"] and d_miou >= -KEEP_RULE["max_miou_drop"])
    gate = m["miou"] > 0.52 and m["iou"][3] > 0.30
    print(f"  {v:>14}: mIoU {m['miou']:.4f} ({d_miou:+.4f}) | Destroyed IoU {m['iou'][3]:.4f} "
          f"({d_destroyed:+.4f}) → {'KEEP' if verdicts[v] else 'not kept'} | Phase Gate {'pass' if gate else 'fail'}")

# 5) Output files
pd.DataFrame(inst_rows).to_csv(OUT_DIR / "instances_val.csv", index=False)
(OUT_DIR / "history.json").write_text(json.dumps(history, indent=1))

CSV_FIELDS = ["member", "experiment_id", "category", "variable", "value", "miou", "bg_iou",
              "intact_iou", "damaged_iou", "destroyed_iou", "precision", "recall", "f1", "f2",
              "destroyed_precision", "destroyed_recall", "destroyed_f2", "train_time_hrs", "seed", "notes"]

def result_row(exp_id, m, value, notes, train_hrs=None):
    return {
        "member": "Phase C (R16)", "experiment_id": exp_id, "category": "E-protocol",
        "variable": "two-stage", "value": value, "miou": round(m["miou"], 4),
        "bg_iou": round(float(m["iou"][0]), 4), "intact_iou": round(float(m["iou"][1]), 4),
        "damaged_iou": round(float(m["iou"][2]), 4), "destroyed_iou": round(float(m["iou"][3]), 4),
        "precision": round(float(m["precision"].mean()), 4), "recall": round(float(m["recall"].mean()), 4),
        "f1": round(float(m["f1"].mean()), 4), "f2": round(float(m["f2"].mean()), 4),
        "destroyed_precision": round(float(m["precision"][3]), 4),
        "destroyed_recall": round(float(m["recall"][3]), 4), "destroyed_f2": round(float(m["f2"][3]), 4),
        "train_time_hrs": train_hrs, "seed": SEED, "notes": notes,
    }

DESCRIBE = {f"{UNET}_std": f"{UNET} pixels (reference)", f"{UNET}_tta": f"{UNET} pixels + 8-view TTA (reference)",
            "R16_sam3": "SAM 3 buildings, grader", f"R16_sam3+{UNET.lower()}": f"SAM 3 + {UNET} pieces, grader",
            "R16_vote": f"SAM 3 + {UNET} pieces, grader/{UNET} vote"}
with open(OUT_DIR / "D4_R16.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)
    writer.writeheader()
    for v in VARIANTS:
        hrs = None if v.startswith(UNET) else round(GRADER_MIN / 60, 2)
        writer.writerow(result_row(v, metrics(cms[v]["all"]), DESCRIBE[v], "E-protocol, last epoch", hrs))

config = {"task": "R16", "smoke_test": SMOKE_TEST, "train_images": len(train_pairs), "val_images": len(val_pairs),
          "sam3_dir": str(SAM3_DIR), "unet": UNET, "unet_checkpoint": str(UNET_CKPT), "unet_reproduces_csv": sanity,
          "crop": CROP, "margin": MARGIN, "min_area": MIN_AREA, "max_intact_per_image": MAX_INTACT_PER_IMAGE,
          "epochs": EPOCHS, "batch": BATCH, "lr": LR, "weight_decay": WEIGHT_DECAY, "seed": SEED,
          "train_buildings": {g: int(c) for g, c in zip(GRADES, counts)},
          "grader_minutes": round(GRADER_MIN, 1), "validation_minutes": round(VAL_MIN, 1),
          "missing_sam3_maps": len(missing_sam), "true_buildings_macro_f1": macro_f1, "keep": verdicts}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1), encoding="utf-8")

(OUT_DIR / "README.md").write_text("\n".join([
    "# R16 results: two-stage model (find each building, then grade it)", "",
    f"{len(val_pairs)} validation images{' (SMOKE TEST)' if SMOKE_TEST else ''}. Stage 1: SAM 3 `pre_grid2` (E3), "
    f"plus {UNET}'s building pieces that SAM 3 missed. Stage 2: Siamese ResNet-18 grader, {CROP} px crops, "
    f"{EPOCHS} epochs, {len(train_grades):,} training buildings.", "",
    "## Pixel scores (E-protocol)", "", as_markdown(overall), "",
    f"{UNET} reproduces its D4 CSV: **{sanity}**.", "",
    f"## Keep rule against {UNET} TTA", "",
    *[f"- {v}: {'KEEP' if k else 'not kept'}" for v, k in verdicts.items()], "",
    "## The grader on the true validation buildings (object level)", "", as_markdown(true_df), "",
    f"Macro F1 {macro_f1} (BDAChat reports 0.763 on its own split, so the comparison is rough).", "",
    "## Files", "",
    "- `D4_R16.csv` (D4 format), `R16_summary.csv` (also per source), `grader_true_buildings.csv`",
    f"- `instances_val.csv`: one row per stage-1 building (the grader's probabilities, {UNET}'s and the truth's pixel counts)",
    f"- `{UNET}_tta/`: {UNET}'s TTA masks (uint8, 0-3, same names as the after-images)",
    "- `grader.pth`, `history.json`, `config.json`", "",
    "## Benchmark use", "",
    "SAM 3 isn't trained and the grader saw DisasterM3's training images only. The U-Net decides the rest: "
    "with R11 (never trained on benchmark pictures) these outputs may answer benchmark questions; "
    "with R12 they may not.",
]), encoding="utf-8")
print(f"\n✓ Wrote {OUT_DIR}: D4_R16.csv, R16_summary.csv, grader_true_buildings.csv, instances_val.csv, "
      "history.json, config.json, README.md")


     variant   miou  background_iou  intact_iou  damaged_iou  destroyed_iou  building_iou  destroyed_precision  destroyed_recall
     R11_std 0.5811          0.9750      0.6753       0.3828         0.2911        0.7356               0.5379            0.3882
     R11_tta 0.5875          0.9759      0.6821       0.3956         0.2965        0.7420               0.5518            0.3906
    R16_sam3 0.4711          0.9685      0.4713       0.2772         0.1672        0.6304               0.1902            0.5813
R16_sam3+r11 0.4941          0.9752      0.5690       0.2800         0.1522        0.7426               0.1600            0.7576
    R16_vote 0.5796          0.9752      0.6846       0.3975         0.2609        0.7426               0.4142            0.4135

R11 reproduces its D4 CSV: yes

The grader on the true validation buildings:
    grade  buildings  precision  recall     f1  graded_intact  graded_damaged  graded_destroyed
   Intact      19624     0.9709  0.7512 0.8470      